In [ ]:
code = 'SREW_RANGE'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output\\'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def SREW_RANGE(bt, start_time, end_time, orderside, sl, intra_sl, om, fixed_or_dynamic, normal_or_cut, synthetic_future, dte1re, dte2re, dte3re, dte4re, dte5re):
    try:
        start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
        end_dt = datetime.datetime.combine(bt.current_week_dates[-1], end_time)
        
        eod_modify = False if fixed_or_dynamic == 'FIXED' else True

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None
        std_ce_scrip, _, std_ce_price, std_pe_price, _, _ = bt.get_strike(start_dt, end_dt, om=0)
        if std_ce_scrip is None: return None
        
        std_strike = int(std_ce_scrip[:-2])
        lower_range, upper_range, intra_lower_range, intra_upper_range = bt.get_sl_range(std_strike, std_ce_price+std_pe_price, sl, intra_sl)

        entry_time = start_dt
        std_sl_flag, std_intra_sl_flag, std_sl_time, day_wise_mtm, _, std_pnl = bt.sl_range_check_combine_leg(start_dt, end_dt, ce_scrip, pe_scrip, lower_range, upper_range, intra_lower_range, intra_upper_range, std_strike, orderside=orderside, eod_modify=eod_modify, range_sl=sl, intra_range_sl=intra_sl, is_on_synthetic=synthetic_future, need_day_wise_mtm=True)
        first_straddle = [start_dt, f"({ce_scrip}, {pe_scrip})", ce_price+pe_price, std_sl_flag, std_intra_sl_flag, std_sl_time, std_pnl]
        
        re_entries_left = {1: dte1re, 2: dte2re, 3: dte3re, 4: dte4re, 5: dte5re}
        
        re_straddles = []
        for re_no in range(max_re):
            
            if std_sl_time and (std_sl_time < end_dt - datetime.timedelta(minutes=5)):

                rdte = int(dte_file.loc[pd.to_datetime(std_sl_time.date()), bt.index])

                if (re_entries_left[rdte] == 0) or (normal_or_cut == 'CUT'):
                    if rdte == 1: # expiry day
                        std_sl_time = ''
                        re_straddles.extend(['', '', '', False, False, '', 0])
                        continue

                    std_sl_time = max(std_sl_time, datetime.datetime.combine(std_sl_time.date(), datetime.time(15,25)))
                else:
                    re_entries_left[rdte] -= 1
            
                start_dt = std_sl_time
                ce_scrip, pe_scrip, ce_price, pe_price, _, start_dt = bt.get_strike(start_dt, end_dt, om=om)
                if ce_scrip is None:
                    std_sl_time = ''
                    re_straddles.extend(['', '', '', False, False, '', 0])
                    continue

                std_ce_scrip, _, std_ce_price, std_pe_price, _, _ = bt.get_strike(start_dt, end_dt, om=0)
                if std_ce_scrip is None:
                    std_sl_time = ''
                    re_straddles.extend(['', '', '', False, False, '', 0])
                    continue
                    
                std_strike = int(std_ce_scrip[:-2])
                lower_range, upper_range, intra_lower_range, intra_upper_range = bt.get_sl_range(std_strike, std_ce_price+std_pe_price, sl, intra_sl)

                std_sl_flag, std_intra_sl_flag, std_sl_time, day_wise_mtm_re, _, std_pnl = bt.sl_range_check_combine_leg(start_dt, end_dt, ce_scrip, pe_scrip, lower_range, upper_range, intra_lower_range, intra_upper_range, std_strike, orderside=orderside, eod_modify=eod_modify, range_sl=sl, intra_range_sl=intra_sl, is_on_synthetic=synthetic_future, need_day_wise_mtm=True)
                day_wise_mtm = {k: day_wise_mtm.get(k, 0) + day_wise_mtm_re.get(k, 0) for k in day_wise_mtm.keys() | day_wise_mtm_re.keys()}
                re_straddles.extend([start_dt, f"({ce_scrip}, {pe_scrip})", ce_price+pe_price, std_sl_flag, std_intra_sl_flag, std_sl_time, std_pnl])
            else:
                re_straddles.extend(['', '', '', False, False, '', 0])
        
        day_wise_mtm = dict(sorted(day_wise_mtm.items()))
        return [code, bt.index, start_time, end_time, orderside, sl, intra_sl, om, fixed_or_dynamic, normal_or_cut, synthetic_future, dte1re, dte2re, dte3re, dte4re, dte5re, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), bt.from_dte, bt.to_dte, len(bt.current_week_dates), entry_time.time(), future_price] + first_straddle + re_straddles + ([0]*5 + list(day_wise_mtm.values()))[-5:]
    
    except Exception as e:
        print(e, [bt.index, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), start_time, end_time, orderside, sl, intra_sl, om, fixed_or_dynamic, normal_or_cut, synthetic_future, dte1re, dte2re, dte3re, dte4re, dte5re])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, from_dte, to_dte, from_date, to_date, start_time, end_time, week_lists = get_meta_row_data(meta_row, pickle_path, weekly=True)
            dte_file = get_dte_file(pickle_path)
            max_re = 20

            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_SL/P_intraSL/P_OM/P_FixedOrDynamic/P_NormalOrCut/P_SyntheticFuture/P_Dte1Re/P_Dte2Re/P_Dte3Re/P_Dte4Re/P_Dte5Re/Start.Date/End.Date/Start.DTE/End.DTE/DayCount/EntryTime/Future'

            for r in range(max_re+1):
                log_cols += f'/STD{r}.Time/STD{r}.Scrip/STD{r}.Open/STD{r}.SL.Flag/STD{r}.IntraSL/STD{r}.SL.Time/STD{r}.PNL'
            log_cols += '/Dte5.PNL/Dte4.PNL/Dte3.PNL/Dte2.PNL/Dte1.PNL'
            log_cols = log_cols.split('/')

            for week_dates in week_lists:
                from_date = week_dates[0]
                to_date = week_dates[-1]

                file_name = f"{index} {week_dates[0].date()} {week_dates[-1].date()} {from_dte}-{to_dte} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    wbt = WeeklyBacktest(pickle_path, index, week_dates, from_dte, to_dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SREW_RANGE(wbt, row['entry_time'], row['exit_time'], row['orderside'], row['sl'], row['intra_sl'], row['om'], row['fixed_or_dynamic'], row['normal_or_cut'], row['synthetic_future'], row['dte1re'], row['dte2re'], row['dte3re'], row['dte4re'], row['dte5re']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                    
                    del wbt
                    t2 = datetime.datetime.now()
                    print(t2-t1)

        except Exception as e:
            input(str(e))